# Lesson 13 | Simulation is not a chip

We can already write **Register-Transfer Level (RTL)**, run testbenches, and inspect waveforms. Simulation tells us whether modeled behavior matches the intended logic.

Today asks one question:

> **Why does a passing simulation not yet prove that a design can run on a Field-Programmable Gate Array (FPGA)?**

Primary new concept: **RTL still has to pass synthesis, implementation, and timing before becoming a configurable hardware image.**

## 1. Concept ledger

**Already known:** RTL, clock, register, testbench, waveform, simulation.

**New today:**
- **synthesis**: transforms RTL into a network implementable with FPGA logic resources;
- **implementation**: maps, places, and routes that network on a specific FPGA;
- **timing analysis**: checks whether signal propagation meets clock-period constraints;
- **bitstream**: configuration data for the FPGA's programmable resources.

**Preview only:** development boards, host communication, external memory, and on-chip communication protocols come later.

## 2. Simulation and synthesis answer different questions

Simulation asks about **behavior**: do state updates, thresholds, resets, and spikes match the test oracle?

Synthesis asks about **implementable structure**: can the RTL become lookup tables, registers, memories, and interconnect?

So “the testbench passes” and “the design can run at the target frequency” require separate evidence.

## 3. From RTL to a bitstream

```mermaid
flowchart LR
  RTL["RTL / SystemVerilog"] --> SIM["simulation"]
  RTL --> SYN["synthesis"]
  SYN --> IMP["implementation"]
  IMP --> TIM["timing analysis"]
  TIM --> BIT["bitstream"]
  BIT --> FPGA["configured FPGA"]
```

The goal is the flow, not memorizing one vendor tool's buttons.

## 4. Run: perform a real synthesis dry run with Yosys

This time Python will not imitate synthesis. The open-source synthesis tool **Yosys** reads the already-simulated `rtl/learning/clocked_accumulator.sv` directly.

Predict first: will Yosys treat SystemVerilog like sequential software, or transform it into hardware structures such as registers, addition, and control logic?

If the next cell says Yosys was not found, follow the [HDL toolchain setup guide](../../docs/en/HDL_TOOLCHAIN_SETUP.md), then restart JupyterLab from the same terminal where the toolchain is activated.

In [ ]:
from pathlib import Path
import re
import shutil
import subprocess

def repo_root():
    for path in (Path.cwd(), *Path.cwd().parents):
        if (path / "rtl" / "learning" / "clocked_accumulator.sv").is_file():
            return path
    raise FileNotFoundError("Run this notebook from inside the FPGA-FlyBrain repository")

root = repo_root()
yosys = shutil.which("yosys")
rtl = root / "rtl" / "learning" / "clocked_accumulator.sv"

if yosys is None:
    print("Yosys was not found.")
    print("Follow docs/en/HDL_TOOLCHAIN_SETUP.md, restart JupyterLab from the activated terminal, and rerun this cell.")
else:
    command = (
        f'read_verilog -sv "{rtl}"; '
        "hierarchy -check -top clocked_accumulator; "
        "proc; opt; stat; check"
    )
    result = subprocess.run(
        [yosys, "-p", command],
        cwd=root,
        check=True,
        text=True,
        capture_output=True,
    )
    yosys_log = result.stdout

    cell_count = None
    cell_types = []
    in_stat = False
    for line in yosys_log.splitlines():
        stripped = line.strip()
        if stripped == "=== clocked_accumulator ===":
            in_stat = True
            continue
        if in_stat:
            match_count_new = re.fullmatch(r"(\d+)\s+cells", stripped)
            match_count_old = re.fullmatch(r"Number of cells:\s*(\d+)", stripped)
            match_cell = re.fullmatch(r"(\d+)\s+(\$\S+)", stripped)
            if match_count_new:
                cell_count = int(match_count_new.group(1))
            elif match_count_old:
                cell_count = int(match_count_old.group(1))
            elif match_cell:
                cell_types.append((match_cell.group(2), int(match_cell.group(1))))
            elif stripped.startswith("Checking module"):
                in_stat = False

    match = re.search(r"Found and reported (\d+) problems", yosys_log)
    problems = int(match.group(1)) if match else None

    print("Yosys synthesis succeeded.")
    print("Top module: clocked_accumulator")
    if cell_count is not None:
        print("Number of synthesized cells:", cell_count)
    if cell_types:
        print("Cell summary:")
        for cell_type, count in cell_types:
            print(f" - {cell_type}: {count}")
    if problems is not None:
        print("Structural check problems:", problems)
    print()
    print("The full log is available in yosys_log; for this lesson, read the teaching summary above first.")


## 5. Observe: this was real synthesis

The most useful part of the summary is not Yosys's internal pass names, but the resulting structure. For this accumulator, you should see arithmetic hardware, a state-holding sequential cell, and the structural-check result.

That evidence answers whether RTL can be transformed into implementable hardware structure. It is **not yet** placement/routing for a specific FPGA, device timing sign-off, or a bitstream. Those require target-platform implementation and timing tools later.

## 6. Why can timing fail?

A register output needs time to travel through combinational logic before the next register samples it. The target clock period acts like a deadline.

For this lesson:

- **critical path**: the longest combinational path delay;
- **slack** = clock period − critical-path delay;
- slack ≥ 0: the teaching model meets timing;
- slack < 0: the target clock is too fast.

Real timing analysis includes setup/hold, clock uncertainty, and additional constraints; those are deferred.

## 7. Run: a minimal timing budget

Predict the critical path and whether a 5 ns clock period can contain it.

In [ ]:
path_delays_ns = [2.2, 4.8, 3.1]
clock_period_ns = 5.0

critical_path_ns = max(path_delays_ns)
slack_ns = clock_period_ns - critical_path_ns
meets_timing = slack_ns >= 0

print(f"critical path: {critical_path_ns:.2f} ns")
print(f"clock period: {clock_period_ns:.2f} ns")
print(f"slack: {slack_ns:+.2f} ns")
print("meets timing:", meets_timing)


## 8. Observe

The longest path is 4.8 ns. With a 5.0 ns target period, the simplified slack is **+0.20 ns**.

This is only a teaching timing model, not a complete device timing sign-off.

## 9. Try It

Change `clock_period_ns` to 4.0. Predict first:

1. Does the critical path change?
2. What happens to the sign of slack?
3. Did the RTL's logical function change?

This separates functional correctness from meeting a timing target.

## 10. Exercise

Open:

[Lesson 13 exercise: read a timing budget](../../exercises/en/13_simulation_is_not_chip.ipynb)

Compute the critical path, slack, and simplified timing result.

## 11. AI Task

Give an AI a synthesis/timing summary and ask it to explain resource usage, critical path, and slack. Check that it does not treat a simulation pass as a timing pass.

## 12. Human Check

Without AI, explain what simulation, synthesis, implementation, and timing analysis each ask; why a correct waveform can still accompany timing failure; how a bitstream relates to RTL; and why no physical board is needed yet.

## 13. Engineering Handoff

This lesson maps to `RMD-011A`: use Yosys for a real synthesis dry run of existing RTL, then read a simplified timing budget. No physical board is required.

## 14. Project Trace

- Lesson: `LSN-013`
- Mapping: `RMD-011A`
- Evidence: synthesis report + timing summary
- Boundary: no physical board required

## 15. Exit Ticket

You can explain why functional simulation and target-clock feasibility require separate verification.